In [1]:
%%writefile main.py
"""
Orbit Wars - "Interstellar Monopoly" Agent

Strategy:
  1. Interest/Production Calculation: Factors in enemy production rates during travel time.
  2. Asset Valuation (ROI): Ranks targets by (Production Value / Cost). Enemy planets 
     are worth double because taking them creates a 2x production swing.
  3. Insider Trading: Reads active fleets to ensure we don't double-spend on a target 
     that is already under acquisition.
  4. Intercept Math: Predicts orbital motion to hit moving targets accurately.
  5. Risk Management: Cancels investments (attacks) if the trajectory hits the sun.
"""

import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

def get_speed(ships):
    """Calculates fleet speed based on ship count using the game's logarithmic curve."""
    if ships <= 1: 
        return 1.0
    ratio = min(1.0, max(0.0, math.log(ships) / math.log(1000)))
    return 1.0 + 5.0 * (ratio ** 1.5)

def intersects_sun(x1, y1, x2, y2, margin=10.5):
    """Checks if the line segment from (x1,y1) to (x2,y2) gets too close to the Sun (50, 50)."""
    px, py = 50.0, 50.0
    l2 = (x2 - x1)**2 + (y2 - y1)**2
    if l2 == 0: 
        return False
    t = max(0, min(1, ((px - x1)*(x2 - x1) + (py - y1)*(y2 - y1)) / l2))
    proj_x = x1 + t * (x2 - x1)
    proj_y = y1 + t * (y2 - y1)
    return math.hypot(px - proj_x, py - proj_y) <= margin

def is_under_acquisition(target, my_fleets, angular_velocity):
    """Checks if we already have a fleet perfectly intercepted to capture this target."""
    for f in my_fleets:
        dist = math.hypot(target.x - f.x, target.y - f.y)
        speed = get_speed(f.ships)
        T = dist / max(1.0, speed)
        
        tx, ty = target.x, target.y
        orb_rad = math.hypot(tx - 50, ty - 50)
        
        # Predict where the target will be when our active fleet arrives
        if orb_rad + target.radius < 50:
            base_angle = math.atan2(ty - 50, tx - 50)
            new_angle = base_angle + angular_velocity * T
            tx = 50 + orb_rad * math.cos(new_angle)
            ty = 50 + orb_rad * math.sin(new_angle)
            
        angle_to_intercept = math.atan2(ty - f.y, tx - f.x)
        
        # If one of our fleets is pointing right at the intercept point, skip re-targeting
        diff = (f.angle - angle_to_intercept + math.pi) % (2 * math.pi) - math.pi
        if abs(diff) < 0.1:
            return True
    return False

def agent(obs):
    moves =[]
    
    # Parse Environment Data
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    angular_velocity = obs.get("angular_velocity", 0.0) if isinstance(obs, dict) else obs.angular_velocity
    comet_ids = obs.get("comet_planet_ids",[]) if isinstance(obs, dict) else obs.comet_planet_ids
    
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets =[Planet(*p) for p in raw_planets]
    
    raw_fleets = obs.get("fleets", []) if isinstance(obs, dict) else obs.fleets
    fleets =[Fleet(*f) for f in raw_fleets]
    
    my_planets =[p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    my_fleets = [f for f in fleets if f.owner == player]

    if not targets:
        return moves

    for mine in my_planets:
        best_target = None
        best_score = -1
        best_cost = 0
        best_angle = 0
        
        for target in targets:
            # 1. Volatile Asset Check: Ignore comets (their paths expire/leave the board)
            if target.id in comet_ids:
                continue
                
            # 2. Insider Ledger Check: Don't spend on an asset we are already acquiring
            if is_under_acquisition(target, my_fleets, angular_velocity):
                continue
                
            # 3. Base Math: Distance and Baseline Fleet Speed
            dist = math.hypot(mine.x - target.x, mine.y - target.y)
            base_cost = target.ships + 1
            speed = get_speed(base_cost)
            T = dist / speed
            
            # 4. Accrued Interest Calculation: Enemies generate ships while we travel
            if target.owner != -1:
                cost = base_cost + (target.production * math.ceil(T))
                # Re-adjust speed/Time because heavier fleets travel faster
                speed = get_speed(cost) 
                T = dist / speed
            else:
                cost = base_cost
                
            # Can our bank account afford the true cost?
            if mine.ships < cost:
                continue
                
            # 5. Perfect Intercept Trajectory (Iterative Orbit Prediction)
            tx, ty = target.x, target.y
            orb_rad = math.hypot(tx - 50, ty - 50)
            is_orbiting = (orb_rad + target.radius < 50)
            
            for _ in range(3): # 3 iterations guarantees high precision intercept
                dist = math.hypot(tx - mine.x, ty - mine.y)
                T = dist / speed
                if is_orbiting:
                    base_angle = math.atan2(target.y - 50, target.x - 50)
                    new_angle = base_angle + angular_velocity * T
                    tx = 50 + orb_rad * math.cos(new_angle)
                    ty = 50 + orb_rad * math.sin(new_angle)
            
            # 6. Bad Investment Check: Does the trajectory hit the sun?
            if intersects_sun(mine.x, mine.y, tx, ty):
                continue
                
            # 7. Monopoly ROI (Return on Investment) Scoring
            # Enemy targets are worth double because taking it = (+X for us, -X for them)
            prod_value = target.production * (2 if target.owner != -1 else 1)
            # We want high production, low acquisition cost, and relatively close proximity
            score = prod_value / (cost + dist * 0.1)
            
            if score > best_score:
                best_score = score
                best_target = target
                best_cost = cost
                best_angle = math.atan2(ty - mine.y, tx - mine.x)
                
        # Execute the trade
        if best_target is not None:
            moves.append([mine.id, best_angle, int(best_cost)])
            # Simulate withdrawing funds to prevent over-drafting if we wanted multiple targets
            mine = Planet(mine.id, mine.owner, mine.x, mine.y, mine.radius, mine.ships - int(best_cost), mine.production)

    return moves

Writing main.py
